# RL Environments

In [1]:
import random
import torch

def setup() -> tuple[torch.Generator, int]:
    if torch.backends.mps.is_available():
        device = torch.device("mps")
        torch.set_default_device(device)
        generator = torch.Generator(device=device)
        print("MPS device found!")
    else:
        device = torch.device("cpu")
        generator = torch.Generator(device=device)
        print("Defaulting to CPU!")

    SEED = random.SystemRandom().randint(0, 2**32 - 1)
    torch.manual_seed(SEED)
    print(f"Random seed: {SEED}")
    return generator, SEED


generator, SEED = setup()

MPS device found!
Random seed: 3595461598


# Raw Environment

In [2]:
import gymnasium as gym

env = gym.make("CartPole-v1")

observation, info = env.reset(seed=SEED)

for _ in range(10):
    action = env.action_space.sample()
    observation, reward, terminated, truncated, info = env.step(action)

    print(observation)

    if terminated or truncated:
        observation, info = env.reset()
env.close()


[ 0.0497654  -0.1858678   0.034694    0.34957173]
[0.04604804 0.00874398 0.04168544 0.0680277 ]
[ 0.04622292  0.20324427  0.04304599 -0.21121731]
[ 0.05028781  0.39772516  0.03882165 -0.4900168 ]
[ 0.05824231  0.20207767  0.02902131 -0.18535599]
[ 0.06228386  0.39677262  0.02531419 -0.46874437]
[ 0.07021932  0.591528    0.0159393  -0.7533422 ]
[ 8.2049869e-02  7.8642660e-01  8.7245705e-04 -1.0409670e+00]
[ 0.0977784   0.9815369  -0.01994688 -1.3333759 ]
[ 0.11740914  0.78667206 -0.0466144  -1.0470008 ]


## TorchRL Environment

In [3]:
from torchrl.envs import GymEnv

env = GymEnv("CartPole-v1")

td = env.reset(seed=SEED)
print(td)
      
for _ in range(10):
    # Random action
    td = env.rand_step(td)

    print(td["next", "observation"])

    # Check whether the episode ended
    if td["next", "done"]:
        td = env.reset()

print(td)
env.close()

TensorDict(
    fields={
        done: Tensor(shape=torch.Size([1]), device=mps:0, dtype=torch.bool, is_shared=True),
        observation: Tensor(shape=torch.Size([4]), device=mps:0, dtype=torch.float32, is_shared=True),
        terminated: Tensor(shape=torch.Size([1]), device=mps:0, dtype=torch.bool, is_shared=True),
        truncated: Tensor(shape=torch.Size([1]), device=mps:0, dtype=torch.bool, is_shared=True)},
    batch_size=torch.Size([]),
    device=None,
    is_shared=False)
tensor([ 0.0498, -0.1859,  0.0347,  0.3496], device='mps:0')
tensor([ 0.0460, -0.3815,  0.0417,  0.6530], device='mps:0')
tensor([ 0.0384, -0.1869,  0.0547,  0.3737], device='mps:0')
tensor([0.0347, 0.0074, 0.0622, 0.0988], device='mps:0')
tensor([ 0.0348,  0.2015,  0.0642, -0.1736], device='mps:0')
tensor([ 0.0389,  0.3957,  0.0607, -0.4454], device='mps:0')
tensor([ 0.0468,  0.1998,  0.0518, -0.1342], device='mps:0')
tensor([0.0508, 0.0039, 0.0491, 0.1744], device='mps:0')
tensor([ 0.0508, -0.1919,  0.052

# TorchRL TensorDictReplayBuffer

In [4]:
from torchrl.envs import GymEnv
from torchrl.data import (
    TensorDictReplayBuffer,
    LazyTensorStorage,
)

sample_batch_size = 16
episode_length = 200
storage_mem_max = episode_length

env = GymEnv("CartPole-v1")
print(f"Environment: {env}")
print(f"Environment batch size: {env.batch_size}")
print(f"Environment observation spec: {env.observation_spec}")
print(f"Environment action spec: {env.action_spec}")
print(f"Environment state spec: {env.state_spec}")

td = env.reset()
replay_buffer = TensorDictReplayBuffer(
    storage=LazyTensorStorage(storage_mem_max),
    batch_size=sample_batch_size,
)

for _ in range(episode_length):

    # Random action for this example
    action = env.action_spec.rand()

    # Put action into current TensorDict
    td["action"] = action

    # Step environment
    td = env.step(td)

    # Store the transition
    replay_buffer.add(td)

    # Continue from next state
    td = td["next"]

    # Reset when episode ends
    if td["done"].item():
        td = env.reset()



batch = replay_buffer.sample(batch_size=10)

print(batch)
print(batch.batch_size)

print("obs:", batch["observation"].shape)
print("action:", batch["action"].shape)
print("reward:", batch["next", "reward"].shape)
print("next obs:", batch["next", "observation"].shape)

Environment: GymEnv(env=CartPole-v1, batch_size=torch.Size([]), device=None)
Environment batch size: torch.Size([])
Environment observation spec: Composite(
    observation: BoundedContinuous(
        shape=torch.Size([4]),
        space=ContinuousBox(
            low=Tensor(shape=torch.Size([4]), device=mps:0, dtype=torch.float32, contiguous=True),
            high=Tensor(shape=torch.Size([4]), device=mps:0, dtype=torch.float32, contiguous=True)),
        device=mps:0,
        dtype=torch.float32,
        domain=continuous),
    device=None,
    shape=torch.Size([]),
    data_cls=None)
Environment action spec: OneHot(
    shape=torch.Size([2]),
    space=CategoricalBox(n=2),
    device=mps:0,
    dtype=torch.int64,
    domain=discrete)
Environment state spec: Composite(device=None, shape=torch.Size([]), data_cls=None)
2026-10-04 10:43:39,548 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([200]) shape [END]
TensorDict(
    fields={
        action: Tensor(shape=torch.S

/Users/ronnie/Local Documents/Non iCloud/Repos/ronniestafford/ai-ml-research/.venv/lib/python3.14/site-packages/torchrl/data/replay_buffers/replay_buffers/base.py:2314: UserWarning: Got conflicting batch_sizes in constructor (16) and `sample` (10). Refer to the ReplayBuffer documentation for a proper usage of the batch-size arguments. The batch-size provided to the sample method will prevail.
  warnings.warn(
